In [1]:
!pip install -q datasets torch evaluate tokenizers transformers rouge_score sacrebleu

# Dataset

In [2]:
from datasets import load_dataset

dataset = load_dataset("knkarthick/dialogsum")
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [3]:
dataset['train'][0]

{'id': 'train_0',
 'dialogue': "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.\n#Person2#: Ok, thanks doctor.",
 'summary': "Mr. Smith'

# Tokenizer and preprocessing

In [4]:
dataset = dataset.remove_columns("id")
dataset

DatasetDict({
    train: Dataset({
        features: ['dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [5]:
import os
from tokenizers import Tokenizer, pre_tokenizers, trainers, models

VOCAB_SIZE = 20000

tokenizer = Tokenizer(models.WordLevel(unk_token="<unk>"))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

trainer = trainers.WordLevelTrainer(
    special_tokens=['<pad>', '<unk>', '<bos>', '<eos>'],
    min_frequency=2,
    vocab_size= VOCAB_SIZE
)

In [6]:
def get_data(dataset):
    for row in dataset:
        yield row['dialogue']

        yield row['summary']

        yield row['topic']

tokenizer.train_from_iterator(get_data(dataset['train']), trainer=trainer)
tokenizer.save('tokenizer.json')

In [7]:
from transformers import PreTrainedTokenizerFast

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file='tokenizer.json',
    unk_token='<unk>',
    pad_token='<pad>',
    bos_token='<bos>',
    eos_token='<eos>',
)

In [8]:
len(tokenizer)

18109

In [9]:
MAX_DIALOGUE_LEN = 256 
MAX_SUMMARY_LEN = 64
MAX_TOPIC_LEN = 25

def preprocess(examples):
    dialogue_text = examples['dialogue']
    summary_text = ['<bos> ' + text + ' <eos>' for text in examples['summary']]
    topic_text = ['<bos> ' + text + ' <eos>' for text in examples['topic']]

    dialogue_encoding = tokenizer(
        dialogue_text,
        padding='max_length',
        truncation=True,
        max_length=MAX_DIALOGUE_LEN
    )
    
    summary_encoding = tokenizer(
        summary_text,
        padding='max_length',
        truncation=True,
        max_length=MAX_SUMMARY_LEN
    )

    topic_encoding = tokenizer(
        topic_text,
        padding='max_length',
        truncation=True,
        max_length=MAX_TOPIC_LEN
    )

    return {
        'input_ids': dialogue_encoding['input_ids'],
        'summary_labels': summary_encoding['input_ids'],
        'topic_labels': topic_encoding['input_ids']
    }

In [10]:
process_dataset = dataset.map(preprocess, batched=True)
process_dataset

DatasetDict({
    train: Dataset({
        features: ['dialogue', 'summary', 'topic', 'input_ids', 'summary_labels', 'topic_labels'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['dialogue', 'summary', 'topic', 'input_ids', 'summary_labels', 'topic_labels'],
        num_rows: 500
    })
    test: Dataset({
        features: ['dialogue', 'summary', 'topic', 'input_ids', 'summary_labels', 'topic_labels'],
        num_rows: 1500
    })
})

In [11]:
print(process_dataset['train'][0])

{'dialogue': "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.\n#Person2#: Ok, thanks doctor.", 'summary': "Mr. Smith's getting a check-u

# Model

In [12]:
from transformers import PreTrainedModel, PretrainedConfig
import torch
import torch.nn as nn

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
device

device(type='cuda')

In [13]:
class Seq2SeqTransformerConfig(PretrainedConfig):
    def __init__(self, vocab_size=20000, max_seq_len=128, embedding_dim=256, n_heads=8, n_layers=6, dropout=0.1, loss_alpha=0.5, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.embedding_dim = embedding_dim
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.dropout = dropout
        self.loss_alpha = loss_alpha

In [14]:
def generate_square_subsequent_mask(sz, device):
    mask = (torch.triu(torch.ones((sz,sz), device=device)) == 1).transpose(0,1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask

def create_mask(src, tgt):
    src_seq_len = src.shape[1]
    tgt_seq_len = tgt.shape[1]
    device = src.device

    tgt_mask = generate_square_subsequent_mask(tgt_seq_len, device)
    src_mask = None 
    
    src_padding_mask = (src == 0)
    tgt_padding_mask = (tgt == 0)
    
    return src_mask, tgt_mask, src_padding_mask, tgt_padding_mask

In [15]:
class Seq2SeqTransformer(PreTrainedModel):
    config_class = Seq2SeqTransformerConfig

    def __init__(self, config, tokenizer):
        super().__init__(config)
        self.embedding = nn.Embedding(config.vocab_size, config.embedding_dim)
        self.pos_embedding = nn.Embedding(config.max_seq_len, config.embedding_dim)

        self.transformer = nn.Transformer(
            d_model = config.embedding_dim,
            nhead = config.n_heads,
            num_encoder_layers = config.n_layers,
            num_decoder_layers = config.n_layers,
            dropout = config.dropout,
            batch_first=True
        )
        self.tokenizer = tokenizer
        self.fc_summary = nn.Linear(config.embedding_dim, config.vocab_size)
        self.fc_topic = nn.Linear(config.embedding_dim, config.vocab_size)
        self.loss = nn.CrossEntropyLoss(ignore_index= tokenizer.pad_token_id)
        self.loss_alpha = config.loss_alpha
    
    def forward(self, input_ids, summary_labels, topic_labels):
        tg_summary_input = summary_labels[:, :-1] 
        tg_topic_input = topic_labels[:, :-1]
        
        tg_summary_output = summary_labels[:, 1:] 
        tg_topic_output = topic_labels[:, 1:]

        # mask
        _, summary_tgt_mask, src_padding_mask, summary_padding_mask = create_mask(input_ids, tg_summary_input)
        _, topic_tgt_mask, _, topic_padding_mask = create_mask(input_ids, tg_topic_input)

        # embedding
        src_seq_len = input_ids.size(1)
        src_positions = torch.arange(0, src_seq_len, dtype=torch.long, device=input_ids.device)

        summary_seq_len = tg_summary_input.size(1)
        summary_positions = torch.arange(0, summary_seq_len, dtype=torch.long, device=input_ids.device)

        topic_seq_len = tg_topic_input.size(1)
        topic_positions = torch.arange(0, topic_seq_len, dtype=torch.long, device=input_ids.device)

        src_embedding = self.embedding(input_ids) + self.pos_embedding(src_positions)
        summary_tgt_embedding = self.embedding(tg_summary_input) + self.pos_embedding(summary_positions)
        topic_tgt_embedding = self.embedding(tg_topic_input) + self.pos_embedding(topic_positions)

        # encoder
        memory = self.transformer.encoder(
            src=src_embedding,
            src_key_padding_mask=src_padding_mask
        )

        # decoder
        summary_out = self.transformer.decoder(
            tgt=summary_tgt_embedding,
            memory=memory,
            tgt_mask=summary_tgt_mask,
            memory_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=summary_padding_mask
        )
        logits_summary = self.fc_summary(summary_out)

        topic_out = self.transformer.decoder(
            tgt=topic_tgt_embedding,
            memory=memory,
            tgt_mask=topic_tgt_mask,
            memory_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=topic_padding_mask
        )
        logits_topic = self.fc_topic(topic_out)

        # loss
        flatten_logits_summary = logits_summary.reshape(-1, logits_summary.size(-1))
        flatten_target_summary = tg_summary_output.reshape(-1)
        
        flatten_logits_topic = logits_topic.reshape(-1, logits_topic.size(-1))
        flatten_target_topic = tg_topic_output.reshape(-1)

        loss_summary = self.loss(flatten_logits_summary, flatten_target_summary)
        loss_topic = self.loss(flatten_logits_topic, flatten_target_topic)
        total_loss =  self.loss_alpha * loss_summary + (1 - self.loss_alpha) * loss_topic

        return {
            "loss": total_loss,
            "loss_summary": loss_summary,
            "loss_topic": loss_topic,
            "logits_summary": logits_summary,
            "logits_topic": logits_topic
        }

In [16]:
config = Seq2SeqTransformerConfig(
    vocab_size=20000, 
    max_seq_len=512, 
    embedding_dim=256, 
    n_heads=8, 
    n_layers=6, 
    dropout=0.1, 
    loss_alpha=0.5
)

model = Seq2SeqTransformer(config, tokenizer)
model.to(device)

Seq2SeqTransformer(
  (embedding): Embedding(20000, 256)
  (pos_embedding): Embedding(512, 256)
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-5): 6 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (linear1): Linear(in_features=256, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=2048, out_features=256, bias=True)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
      (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    )
    (decoder): TransformerDecoder(
      (layers): ModuleList(
    

In [17]:
inputs_id = torch.tensor(process_dataset['train'][0]['input_ids']).unsqueeze(0)
summary_label = torch.tensor(process_dataset['train'][0]['summary_labels']).unsqueeze(0)
topic_label =  torch.tensor(process_dataset['train'][0]['topic_labels']).unsqueeze(0)

inputs_id.shape, summary_label.shape, topic_label.shape

(torch.Size([1, 256]), torch.Size([1, 64]), torch.Size([1, 25]))

In [18]:
output = model(inputs_id.to(device), summary_label.to(device), topic_label.to(device))
output, output['logits_summary'].shape, output['logits_topic'].shape

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


({'loss': tensor(10.1526, device='cuda:0', grad_fn=<AddBackward0>),
  'loss_summary': tensor(10.2261, device='cuda:0', grad_fn=<NllLossBackward0>),
  'loss_topic': tensor(10.0791, device='cuda:0', grad_fn=<NllLossBackward0>),
  'logits_summary': tensor([[[ 0.9806, -1.0666, -0.3669,  ...,  0.5218, -0.8647, -0.1442],
           [ 0.1247, -1.2117, -0.4130,  ...,  0.5047, -1.4773, -0.3776],
           [ 0.6263, -0.7551, -0.6883,  ...,  0.4247, -0.8031, -0.7247],
           ...,
           [ 0.3165, -1.3558, -0.7759,  ...,  0.4740, -1.1781, -0.4509],
           [ 0.7061, -0.7315, -0.8735,  ...,  0.8388, -1.1826, -0.2409],
           [ 0.0646, -1.1210, -0.5684,  ...,  0.6598, -0.5545, -0.4439]]],
         device='cuda:0', grad_fn=<ViewBackward0>),
  'logits_topic': tensor([[[ 0.5034, -0.1212,  0.4957,  ..., -0.3195,  0.3470, -0.2363],
           [ 0.2427, -0.2041,  0.7445,  ...,  0.3740,  0.0497, -0.0266],
           [ 0.1091, -0.3501,  0.2203,  ..., -0.0014,  0.3144, -0.3732],
           ..

# Train

In [19]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir='./res_transformer',
    logging_strategy='epoch',
    eval_strategy='epoch',
    save_strategy='epoch',
    num_train_epochs=50,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    save_total_limit=2,

    fp16=True,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=process_dataset['train'],
    eval_dataset=process_dataset['validation'],
)

In [20]:
trainer.train()

In [21]:
from huggingface_hub import notebook_login
# notebook_login()

In [22]:
repo_id = "huynoob1312/dialogsum-multi-task-transformer"
# tokenizer.push_to_hub(repo_id)
# model.push_to_hub(repo_id)

# Evaluate

In [23]:
from transformers import AutoTokenizer
import evaluate
from tqdm import tqdm

repo_id = "huynoob1312/dialogsum-multi-task-transformer"

In [24]:
tokenizer = AutoTokenizer.from_pretrained(repo_id)

In [25]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import torch

config = Seq2SeqTransformerConfig.from_pretrained(repo_id)
model = Seq2SeqTransformer(config, tokenizer)

weights_path = hf_hub_download(repo_id=repo_id, filename="model.safetensors")
state_dict = load_file(weights_path)
model.load_state_dict(state_dict)
model.to(device)

Seq2SeqTransformer(
  (embedding): Embedding(20000, 256)
  (pos_embedding): Embedding(512, 256)
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-5): 6 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (linear1): Linear(in_features=256, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=2048, out_features=256, bias=True)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
      (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    )
    (decoder): TransformerDecoder(
      (layers): ModuleList(
    

In [26]:
rouge_metric = evaluate.load('rouge')
bleu_metric = evaluate.load('sacrebleu')

In [27]:
def generate_text(model, tokenizer, text, max_summary_len=64, max_topic_len=25):
    model.eval()
    device = next(model.parameters()).device
    
    with torch.no_grad():
        input_encoding = tokenizer(
            text, padding='max_length', truncation=True, 
            max_length=256, return_tensors='pt'
        )
        input_ids = input_encoding['input_ids'].to(device)
        
        # encoder
        src_seq_len = input_ids.size(1)
        src_positions = torch.arange(0, src_seq_len, dtype=torch.long, device=device).unsqueeze(0)
        
        src_emb = model.embedding(input_ids) + model.pos_embedding(src_positions)
        src_padding_mask = (input_ids == tokenizer.pad_token_id)
        
        memory = model.transformer.encoder(
            src=src_emb,
            src_key_padding_mask=src_padding_mask
        )
        
        # genarate summary
        summary_ids = [tokenizer.bos_token_id]
        
        for _ in range(max_summary_len):
            tgt_tensor = torch.tensor([summary_ids], dtype=torch.long, device=device)
            tgt_positions = torch.arange(0, tgt_tensor.size(1), dtype=torch.long, device=device).unsqueeze(0)
            
            tgt_emb = model.embedding(tgt_tensor) + model.pos_embedding(tgt_positions)
            tgt_mask = generate_square_subsequent_mask(tgt_tensor.size(1), device)
            
            out = model.transformer.decoder(
                tgt=tgt_emb,
                memory=memory,
                tgt_mask=tgt_mask,
                memory_key_padding_mask=src_padding_mask
            )
            
            logits = model.fc_summary(out)
            
            next_word = logits[0, -1, :].argmax().item()
            summary_ids.append(next_word)
            
            if next_word == tokenizer.eos_token_id:
                break
                
        # genarate topic
        topic_ids = [tokenizer.bos_token_id]
        
        for _ in range(max_topic_len):
            tgt_tensor = torch.tensor([topic_ids], dtype=torch.long, device=device)
            tgt_positions = torch.arange(0, tgt_tensor.size(1), dtype=torch.long, device=device).unsqueeze(0)
            
            tgt_emb = model.embedding(tgt_tensor) + model.pos_embedding(tgt_positions)
            tgt_mask = generate_square_subsequent_mask(tgt_tensor.size(1), device)
            
            out = model.transformer.decoder(
                tgt=tgt_emb,
                memory=memory,
                tgt_mask=tgt_mask,
                memory_key_padding_mask=src_padding_mask
            )
            
            logits = model.fc_topic(out)
            
            next_word = logits[0, -1, :].argmax().item()
            topic_ids.append(next_word)
            
            if next_word == tokenizer.eos_token_id:
                break
                
    final_summary = tokenizer.decode(summary_ids, skip_special_tokens=True)
    final_topic = tokenizer.decode(topic_ids, skip_special_tokens=True)
    
    return final_summary, final_topic


In [33]:
sample_idx = 10 
test_dialogue = dataset['test'][sample_idx]['dialogue']
real_summary = dataset['test'][sample_idx]['summary']
real_topic = dataset['test'][sample_idx]['topic']

print("HỘI THOẠI GỐC")
print(test_dialogue)
print("\n")

pred_summary, pred_topic = generate_text(model, tokenizer, test_dialogue)

print("KẾT QUẢ CỦA MODEL")
print(f"TOPIC (AI đoán):  {pred_topic}")
print(f"SUMMARY (AI đoán): {pred_summary}")
print("\n")
print("KẾT QUẢ THỰC TẾ")
print(f"TOPIC (Gốc):  {real_topic}")
print(f"SUMMARY (Gốc): {real_summary}")


========= HỘI THOẠI GỐC =========
#Person1#: Happy Birthday, this is for you, Brian.
#Person2#: I'm so happy you remember, please come in and enjoy the party. Everyone's here, I'm sure you have a good time.
#Person1#: Brian, may I have a pleasure to have a dance with you?
#Person2#: Ok.
#Person1#: This is really wonderful party.
#Person2#: Yes, you are always popular with everyone. and you look very pretty today.
#Person1#: Thanks, that's very kind of you to say. I hope my necklace goes with my dress, and they both make me look good I feel.
#Person2#: You look great, you are absolutely glowing.
#Person1#: Thanks, this is a fine party. We should have a drink together to celebrate your birthday


========= KẾT QUẢ CỦA MODEL =========
🔹 TOPIC (AI đoán):  ask for help
🔹 SUMMARY (AI đoán): # Person1 # wants to buy a new refrigerator . # Person2 # recommends one and asks # Person1 # to take it .


========= KẾT QUẢ THỰC TẾ =========
🔸 TOPIC (Gốc):  birthday party
🔸 SUMMARY (Gốc): #Person1# a

In [29]:
pred_summaries = []
ref_summaries = []
pred_topics = []
ref_topics = []

test_data = process_dataset['test']
NUM_TEST = len(test_data)

for i in tqdm(range(NUM_TEST)):
    dialogue = test_data[i]['dialogue']
    
    pred_sum, pred_top = generate_text(model, tokenizer, dialogue)
    
    pred_summaries.append(pred_sum)
    pred_topics.append(pred_top)
    
    ref_summaries.append(test_data[i]['summary'])
    ref_topics.append(test_data[i]['topic'])

summary_rouge = rouge_metric.compute(predictions=pred_summaries, references=ref_summaries)
summary_bleu = bleu_metric.compute(predictions=pred_summaries, references=ref_summaries)
topic_bleu = bleu_metric.compute(predictions=pred_topics, references=ref_topics)
topic_rouge = rouge_metric.compute(predictions=pred_topics, references=ref_topics)


print("\n KẾT QUẢ SUMMARY:")
print(f" - ROUGE-1 (Khớp từng từ): {summary_rouge['rouge1'] * 100:.2f}")
print(f" - ROUGE-2 (Khớp cụm 2 từ): {summary_rouge['rouge2'] * 100:.2f}")
print(f" - ROUGE-L (Khớp chuỗi dài nhất): {summary_rouge['rougeL'] * 100:.2f}")
print(f" - Điểm BLEU của Summary: {summary_bleu['score']:.2f} ")

print("\n KẾT QUẢ TOPIC:")
print(f" - Điểm BLEU: {topic_bleu['score']:.2f}")
print(f" - Điểm ROUGE-L của Topic: {topic_rouge['rougeL'] * 100:.2f}")


100%|██████████| 1500/1500 [06:12<00:00,  4.03it/s]



 KẾT QUẢ SUMMARY (ROUGE):
 - ROUGE-1 (Khớp từng từ): 24.22
 - ROUGE-2 (Khớp cụm 2 từ): 4.07
 - ROUGE-L (Khớp chuỗi dài nhất): 19.97
 - Điểm BLEU của Summary: 10.92 

 KẾT QUẢ TOPIC (BLEU):
 - Điểm BLEU: 2.63
 - Điểm ROUGE-L của Topic: 6.09
